In [85]:
import torch
import torch.nn.functional as F

In [86]:
def get_data(n_users, n_items):
    A = torch.randint(0, 5, (n_users, n_items), dtype=torch.float)
    A[A < 2] = 0 # to make it sparse
    return A

In [87]:
n_users = 100
n_items = 15

In [88]:
data = get_data(n_users, n_items)

In [89]:
def pearson_correaltion(A):
    A = A.T
    rated = (A != 0).float() #mask weather rated or not rated
    # denominator making sure we only take rated entries into considration
    user_means = (A * rated).sum(dim=1) / rated.sum(dim=1).clamp(min=1)
    centered_A = A - user_means.unsqueeze(1)
    centered_A *= rated #reset unrated entries to 0

    # compute the covariance
    numerator = centered_A @ centered_A.T
    row_norms = torch.norm(centered_A, p=2, dim=1, keepdim=True)
    denominator = row_norms @ row_norms.T
    denominator = denominator.clamp(min=1e-6)
    similarity_matrix = numerator / denominator
    return torch.clamp(similarity_matrix, -1.0, 1.0)

In [90]:
similarity_matrix = pearson_correaltion(data)

In [91]:
def find_k(similarity_matrix, item_id, k):
    scores = similarity_matrix.T[item_id]
    scores[item_id] = -2.0 # exclude itself

    return torch.topk(scores, k).indices


In [92]:
find_k(similarity_matrix, 2, 2)

tensor([12, 13])

In [ ]:
def recommend_items(rating, similarity_matrix, top_k, target):
    result = {}
    items_to_consider = torch.where(rating[target] == 0)[0] #items not yet rated by target user
    user_ratings = rating[target]

    for idx in items_to_consider:
        numerator_sum  = 0
        denominator_sum = 0
        # Find items rated by the target user
        rated_items = torch.where(user_ratings != 0)[0]
        for item_idx in rated_items:
            #get ratings for this item from
            similarity_score =  similarity_matrix[item_idx, idx]
            if (similarity_score != 0):
                numerator_sum += similarity_score * user_ratings[item_idx]
                denominator_sum += abs(similarity_score)

        if (denominator_sum > 0):
            predicted_rating =  (numerator_sum / denominator_sum)
            result[idx.item()] = predicted_rating.item()


    return sorted(result.items(), key=lambda x: x[1], reverse=True)


In [94]:
target = 10
top_k = find_k(similarity_matrix, target, 5)

recommend_items(data, similarity_matrix, top_k, target)

[(6, 1.1667927503585815),
 (12, 0.9559764266014099),
 (11, 0.5404044985771179),
 (9, -1.066025733947754),
 (1, -1.3687511682510376),
 (8, -2.474337577819824)]